In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Load the final integrated MEDUSA dataset
df = pd.read_csv('/home/rt334/Downloads/MEDUSA_Master_ULTIMATE.csv', low_memory=False)

# Set up MTAP status and convert it into readable group labels
mtap_col = 'DriverLoss 9p21 (CDKN2A,MTAP,IFNA)'
df[mtap_col] = pd.to_numeric(df[mtap_col], errors='coerce')
df['MTAP_status'] = df[mtap_col].map({0.0: 'Intact', 1.0: 'Loss'})

/tmp/ipykernel_4037/1769772845.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['MTAP_status'] = df[mtap_col].map({0.0: 'Intact', 1.0: 'Loss'})


In [6]:
# These variables separate mutations already present in the tumour from those
# acquired later, so I can compare the evolutionary pattern by MTAP status
evo_cols = ['Number of clonal mutations', 'Number of subclonal mutations',
            'Number of clonal coding mutations', 'Number of subclonal coding mutations',
            'Clonal TMB', 'Subclonal TMB',
            'Number of clonal mutations with neoantigen', 'Number of subclonal mutations with neoantigen',
            'Number of neoantigen peptides from clonal mutations', 'Number of neoantigen peptides from subclonal mutations']

# Add a subclonal-to-clonal ratio as a simple way of looking at the balance
# between later subclonal changes and the clonal mutation burden
df['Subclonal_Clonal_ratio'] = (
    pd.to_numeric(df['Number of subclonal mutations'], errors='coerce') /
    pd.to_numeric(df['Number of clonal mutations'], errors='coerce').replace(0, np.nan)
)

evo_cols.append('Subclonal_Clonal_ratio')

/tmp/ipykernel_4037/1821128549.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Subclonal_Clonal_ratio'] = (


In [7]:
# Compare each evolutionary feature between MTAP-intact and MTAP-loss tumours
# and correct the full set of tests for multiple comparisons
results = []

for c in evo_cols:
    vals = pd.to_numeric(df[c], errors='coerce')
    sub = pd.DataFrame({'v': vals, 'grp': df['MTAP_status']}).dropna()

    intact = sub[sub['grp'] == 'Intact']['v']
    loss = sub[sub['grp'] == 'Loss']['v']

    # Skip any comparison where either group is too small
    if len(intact) < 5 or len(loss) < 5:
        continue

    stat, p = mannwhitneyu(intact, loss)

    # Keep the group sizes, medians and p-value for each feature
    results.append({
        'Feature': c,
        'n(Intact)': len(intact),
        'n(Loss)': len(loss),
        'Median(Intact)': round(intact.median(), 3),
        'Median(Loss)': round(loss.median(), 3),
        'p': p,
    })

# Apply FDR correction across all evolutionary comparisons
res_df = pd.DataFrame(results)
res_df['FDR'] = multipletests(res_df['p'], method='fdr_bh')[1]
res_df = res_df.sort_values('p')

# Save the full results table
res_df.to_csv('table_evolutionary_architecture_vs_mtap.csv', index=False)

# Quick summary of how many features remain significant after correction
print(f"Tested {len(res_df)} evolutionary-timing features vs MTAP status")
print(f"FDR<0.05: {(res_df['FDR']<0.05).sum()} of {len(res_df)}")
print(res_df.to_string(index=False))

Tested 11 evolutionary-timing features vs MTAP status
FDR<0.05: 7 of 11
                                               Feature  n(Intact)  n(Loss)  Median(Intact)  Median(Loss)        p      FDR
         Number of subclonal mutations with neoantigen         56       80           2.000         8.000 0.001289 0.007766
Number of neoantigen peptides from subclonal mutations         56       80           4.000        20.000 0.001412 0.007766
                  Number of subclonal coding mutations         56       80           7.500        23.500 0.005840 0.014681
   Number of neoantigen peptides from clonal mutations         56       80          14.500        17.000 0.006551 0.014681
                                         Subclonal TMB         56       80           0.207         0.592 0.006673 0.014681
                         Number of subclonal mutations         56       80          23.000        83.000 0.009252 0.016963
            Number of clonal mutations with neoantigen         56  

In [8]:
# Plot the main clonal and subclonal features together so the MTAP pattern is easy to compare
MTAP_COLORS = {'Intact': 'steelblue', 'Loss': 'darkorange'}

# Turn the p-values into simple significance labels for the figure
def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return 'n.s.'

# Reuse the same plotting setup for each evolutionary feature
def violin_panel(ax, col, label):
    sub = df[[col, 'MTAP_status']].copy()
    sub[col] = pd.to_numeric(sub[col], errors='coerce')
    sub = sub.dropna()

    intact = sub[sub['MTAP_status'] == 'Intact'][col].values
    loss = sub[sub['MTAP_status'] == 'Loss'][col].values

    # Show the distributions for the two MTAP groups
    parts = ax.violinplot([intact, loss], positions=[1, 2], showmedians=True, widths=0.7)

    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(MTAP_COLORS[['Intact', 'Loss'][i]])
        pc.set_alpha(0.8)
        pc.set_edgecolor('black')
        pc.set_linewidth(0.6)

    for k in ['cmedians', 'cmins', 'cmaxes', 'cbars']:
        parts[k].set_color('black')
        parts[k].set_linewidth(0.8)

    # Add the individual patient values with a small amount of jitter
    rng = np.random.default_rng(0)

    for i, g in enumerate([intact, loss]):
        jitter = rng.normal(0, 0.05, size=len(g))
        ax.scatter(
            np.full(len(g), i + 1) + jitter,
            g,
            color='black',
            alpha=0.3,
            s=8,
            zorder=3
        )

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Intact', 'Loss'])
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_ylabel('Ratio' if 'ratio' in label.lower() else 'Count', fontsize=8)

    # Run the same Mann-Whitney comparison used in the main analysis
    _, p = mannwhitneyu(intact, loss)

    # Add the significance marker above the two groups
    top = max(intact.max(), loss.max())
    ax.set_ylim(ax.get_ylim()[0], top * 1.25)
    ax.plot(
        [1, 1, 2, 2],
        [top * 1.05, top * 1.1, top * 1.1, top * 1.05],
        color='black',
        lw=0.8
    )
    ax.text(1.5, top * 1.12, sig_stars(p), ha='center', fontsize=8)

    # Put the available sample size underneath each group
    y0 = ax.get_ylim()[0] + (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.02
    ax.text(1, y0, f'n={len(intact)}', ha='center', fontsize=6)
    ax.text(2, y0, f'n={len(loss)}', ha='center', fontsize=6)

    ax.set_xlabel('MTAP Status', fontsize=8)

# Features chosen to show the subclonal pattern alongside a clonal comparison
panels = [
    ('Number of subclonal mutations', 'Subclonal mutations'),
    ('Number of subclonal mutations with neoantigen', 'Subclonal mutations\nwith neoantigen'),
    ('Number of neoantigen peptides from subclonal mutations', 'Subclonal neoantigen\npeptides'),
    ('Subclonal TMB', 'Subclonal TMB'),
    ('Number of clonal mutations', 'Clonal mutations'),
    ('Subclonal_Clonal_ratio', 'Subclonal:Clonal\nmutation ratio'),
]

# Build the six-panel figure
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, (col, label) in zip(axes, panels):
    violin_panel(ax, col, label)

# Add the overall figure title and save the final version
fig.suptitle(
    'Clonal vs subclonal mutation architecture by MTAP status',
    fontsize=13,
    y=1.01
)

plt.tight_layout()
plt.savefig(
    'figure_evolutionary_architecture_mtap.png',
    dpi=300,
    bbox_inches='tight'
)
plt.close()

print('\nsaved figure_evolutionary_architecture_mtap.png')


saved figure_evolutionary_architecture_mtap.png
